# How do you score a match between two lists of numbers?

Your music app just played you a song by a band you have never heard of, and it fit. Somewhere in a data center, two lists of numbers got compared, and the comparison said: same taste.

We are going to build that comparison ourselves, one line of NumPy at a time. No hidden magic, just multiply and add.

In [ ]:
import numpy as np

A taste is an arrow. Each entry is a quality, and the size of the entry is how hard the song leans that way.

We'll score one probe arrow (you) against four candidate arrows (songs).

Before you run this: call it out loud. Which candidate do you think ends up with the highest score, and can any score go negative?

In [ ]:
probe = np.array([3, 4])

candidates = {
    "same_genre": np.array([6, 8]),
    "related": np.array([4, 1]),
    "orthogonal": np.array([4, -3]),
    "opposite": np.array([-3, -4]),
}

for name, vec in candidates.items():
    print(name, vec)

Here is the rule: multiply the two arrows entry by entry, then add up the products. One multiply, one add.

Before you run this: use the rule by hand on `same_genre` and `opposite`. What do you get?

In [ ]:
def dot_by_hand(a, b):
    total = 0
    for ai, bi in zip(a, b):
        total += ai * bi
    return total

scores = {name: dot_by_hand(probe, vec) for name, vec in candidates.items()}
for name, score in scores.items():
    print(name, score)

In [ ]:
for name, vec in candidates.items():
    assert dot_by_hand(probe, vec) == np.dot(probe, vec)

print("Manual multiply-and-add matches np.dot exactly.")

`same_genre` wins, no surprise there. But look at `opposite`: -25. The tempting guess was that a score can't go below zero, because how could similarity be less than nothing? It just went to the basement.

The number measures which way the arrows point, not how similar the songs "look."

Now turn the probe instead of picking new candidates. We'll rotate the probe arrow through five angles and score it against one fixed candidate.

Before you run this: predict whether the candidate arrow moves at all while the probe turns. Predict whether the score climbs, falls, or does both.

In [ ]:
def rotate(vec, degrees):
    theta = np.deg2rad(degrees)
    rotation = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)],
    ])
    return rotation @ vec

fixed_candidate = np.array([5, 0])
base_probe = np.array([4, 0])

for degrees in [0, 45, 90, 135, 180]:
    turned_probe = rotate(base_probe, degrees)
    score = np.dot(turned_probe, fixed_candidate)
    print(f"probe at {degrees:>3} degrees -> score {score:6.2f}")

assert np.array_equal(fixed_candidate, np.array([5, 0]))

The candidate never moved. The `assert` above just checked it. Only the probe's direction changed, and the score climbed, died, then went negative anyway.

So a big score does not mean two songs are the same. It means their directions agree.

## Name the machine you just used

Point the probe along a candidate and the score climbs to its biggest. Point it across and the score dies to zero. Point it against and the score goes negative. One multiply and one add, and it behaves like a meter with agree, ignore, and oppose on the dial.

**The dot product is a similarity meter.**

Two lists of numbers go in. One honest reading of alignment comes out.

In [ ]:
along = rotate(base_probe, 0)
across = rotate(base_probe, 90)
against = rotate(base_probe, 180)

score_along = np.dot(along, fixed_candidate)
score_across = np.dot(across, fixed_candidate)
score_against = np.dot(against, fixed_candidate)

print("agree :", score_along)
print("ignore:", score_across)
print("oppose:", score_against)

assert score_along > 0
assert np.isclose(score_across, 0)
assert score_against < 0
assert score_along > score_across > score_against

If the dot product were not a similarity meter, one of those asserts would have blown up. It didn't. Agree is positive, ignore is zero, oppose is negative, in that order, every time.

Before you run this next part: stretch one candidate to be five times longer, but do not turn it at all. Predict whether its score against the probe changes.

In [ ]:
def unit(vec):
    return vec / np.linalg.norm(vec)

original = candidates["related"]
stretched = original * 5

score_original = np.dot(probe, original)
score_stretched = np.dot(probe, stretched)

print("original score :", score_original)
print("stretched score:", score_stretched)

assert np.allclose(unit(original), unit(stretched))
assert score_stretched > score_original

Same direction, confirmed by the first assert: the unit vectors match exactly. Different score anyway, confirmed by the second assert.

That's the flaw shipped on purpose. The meter cannot tell "points the same way" apart from "just bigger." A long, loud song can win on sheer size, not on agreement. A shout beats a whisper even when the whisper agrees with you more.

### For the walk home

What is the cheapest fix that keeps the direction and forgets the size?

Try coding it yourself before next time: write a function that scores two arrows the same way we did here, but stays put when you stretch a candidate longer. Test it the same way we tested the meter, with a candidate and its five-times-longer twin, and an assert that the two scores now come out equal.